In [ ]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI
import os
from urllib.request import urlopen, Request
import base64
from pathlib import Path
from dotenv import load_dotenv

# Add references
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

In [ ]:
# Get configuration settings 
load_dotenv()
openai_endpoint = os.getenv("ENDPOINT")
model_deployment =  os.getenv("MODEL_DEPLOYMENT")


# Create an OpenAI client
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=openai_endpoint,
    api_key=token_provider()
) 
chat_completion = client.chat.completions.create(
    model=model_deployment,
    messages=[{"role": "user", "content": "who is prime minester of india"}]
) 
print(chat_completion.choices[0].message.content)  

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
documents = SimpleDirectoryReader(
    input_files=[r"C:\Users\a855099\Documents\GitHub\mslearn-ai-vision\Labfiles\gen-ai-vision\metagpt.pdf"]
).load_data()

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

In [ ]:
embedding_deployment = os.getenv("EMBEDDING_MODEL_DEPLOYMENT")

In [32]:
from llama_index.core import Settings
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding



In [33]:
Settings.llm = AzureOpenAI(engine=model_deployment,
                           model="gpt-5-mini",
                           azure_endpoint=openai_endpoint.replace("/openai/v1", ""),
                           azure_ad_token_provider=token_provider,
                           use_azure_ad=True,
                           api_version="2024-06-01"
                           )

In [34]:
Settings.embed_model = AzureOpenAIEmbedding(azure_deployment=embedding_deployment,
                                            model="text-embedding-3-small",
                                            azure_endpoint=openai_endpoint.replace("/openai/v1", ""),
                                            azure_ad_token_provider=token_provider,
                                            use_azure_ad=True,
                                            api_version="2024-06-01"
                                       )


In [28]:
embedding = Settings.embed_model.get_text_embedding("Hello world")
print("Embedding length:", len(embedding))

Embedding length: 1536


In [35]:
response = Settings.llm.complete(
"Say hello"
)
print(response)

Hello! How can I help you today?


In [37]:
from llama_index.core import SummaryIndex, VectorStoreIndex

summary_index = SummaryIndex(nodes)
vector_index = VectorStoreIndex(nodes)

In [38]:
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
)
vector_query_engine = vector_index.as_query_engine()

In [39]:
from llama_index.core.tools import QueryEngineTool


summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description=(
        "Useful for summarization questions related to MetaGPT"
    ),
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description=(
        "Useful for retrieving specific context from the MetaGPT paper."
    ),
)

In [40]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector


query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        summary_tool,
        vector_tool,
    ],
    verbose=True
)

In [45]:
response = query_engine.query("could you please explain AGENTS IN STANDARD OPERATING PROCEDURES?")
print(str(response))

Selecting query engine 1: The question requests an explanation of a specific topic likely discussed in the MetaGPT paper, so the option for retrieving specific context from the paper is most relevant..
I can help, but the excerpt I can see is mostly low-level PDF annotation data and citation links (examples include references to autogpt_github, agentverse_github, LangChain, Qian 2023, and Austin 2021). There’s no readable explanatory text in these snippets that defines or explains “agents in standard operating procedures.”

Would you please either
- paste the section or paragraph that discusses “agents in standard operating procedures,” or
- tell me the page or figure/table number in the PDF to look up,

and I’ll summarize and explain it clearly. If you prefer, I can also outline typical things to expect in such a section (roles agents play in SOPs, how they’re governed, examples, risks and mitigations) once you confirm you want that general outline.


In [1]:
print(len(response.source_nodes))

NameError: name 'response' is not defined

In [46]:
response = query_engine.query(
    "How do agents share information with other agents?"
)
print(str(response))

Selecting query engine 1: The question asks for a specific mechanism (how agents share information), which is best answered by retrieving detailed context from the MetaGPT paper rather than a general summary..
They exchange information explicitly and implicitly using a few common mechanisms:
- Direct message passing (agent-to-agent communication).  
- Shared environment or shared memory/state where observations or beliefs are written and read.  
- Passing artifacts or tool outputs (files, API/tool calls, code or results) that other agents can consume.  
- Protocols or coordination patterns (e.g., publish/subscribe, turn-taking, request–response) to structure exchanges.
